# Mühlenanwendungsfall — Kontextanalyse

Eigene Stufe zwischen Modellierung und Drift-Erkennung. Sie analysiert Daten und
Modellgüte in der Reihenfolge von Masterarbeit (Igel 2024, Kap. 4.1–4.3) und Exposé
(Abschnitt 5.2) und liefert am Ende die Ereignisreferenz für *mill_detection*.

| Abschnitt | Frage | Vorlage |
|---|---|---|
| 1 | Hängt der Fehler an der Streuung der Betriebspunkte? | MA 4.1, Abb. 4.1/4.2 |
| 2 | Wie wandern die Stellgrößen, und wie der Fehler mit ihnen? | MA Abb. 4.3, Exposé Abb. 5 |
| 3 | Welche Signale hängen mit dem Fehler zusammen? | MA 4.2.1/4.2.2, Abb. 4.4/4.5 |
| 4 | Fallen Kalibrierungen mit Änderungen der Modellgüte zusammen? | MA 4.2.3, Abb. 4.6, Exposé Abb. 6 |
| 5 | Welche Driftart liegt vor? | MA 4.3 |
| 6 | Ist der Drift global oder lokal im Merkmalsraum? | MA 4.3.1/4.3.2, Abb. 4.9/4.10 |
| 7 | Springt oder wandert der Fehler? | MA 4.4.4 |
| 8 | Ereignisreferenz für die Detektion | — |

**Was sich gegenüber der Masterarbeit geändert hat:** Die Kalibrierungen sind keine
aus dem Offsetsignal geschätzten Zeitpunkte mehr, sondern protokollierte
Nullsetzungen der Walzen-Offsets durch die Steuerung (`data_calibrations` aus
*mill_data*). Damit entfällt die Sprungerkennung im Offsetkanal. Die Materialfeuchte
(`A53-2N01.QZ01`) und die Wassereindüsung (`A56-WI01.FC01`) stehen jetzt im Strom.
Nicht übernommen: der Vergleich mit einem auf Produktionsdaten trainierten Modell
(MA Abb. 4.7) und das 60-Cluster-Modell (MA Abb. 4.8).

In [1]:
FORCE_RECOMPUTE = False
IS_FINAL = False

In [2]:
RUN_ID = "e63cf80b" # None      # None -> neuestes data_scored_stream aus mill_model
SEED = 42

# Plausibilitaetsgrenzen je Signal; Werte ausserhalb sind Sensorfehler.
PLAUSIBLE = {"feed_moisture": (0.0, 25.0), "roller_temperature": (-20.0, 150.0)}
CLIP_LOWER = {"water_injection": 0.0}   # Nullpunktversatz des Durchflussmessers

VAR_WIN_DAYS = 7        # Fenster der Betriebspunktstreuung rolling_var
TREND_WIN_DAYS = 14     # gleitendes Mittel der Stellgroessen
CORR_WIN_DAYS = 90      # gleitende Korrelation Stellgroesse/Fehler
STOP_GAP_H = 2.0        # Luecke zum Vorgaenger, ab der ein Stillstand zaehlt
STOP_WIN_DAYS = 7       # Zaehlfenster fuer Stillstaende
RMSE_WIN_DAYS = 14      # gleitender RMSE

CAL_WIN_DAYS = 7        # Fenster vor/nach einer Kalibrierung
CAL_MIN_POINTS = 10     # Mindestpunkte je Seite
N_PERM = 2000           # Zufallszeitpunkte fuer das Vergleichsniveau

KS_ALPHA = 0.05
RADIUS_FRAC = 0.5       # Clusterradius als Anteil des Abstands zum naechsten Zentrum
OCC_WIN_DAYS = 30       # Fenster der Cluster-Belegung
SAVGOL_WIN_DAYS = 9
SAVGOL_ORDER = 2

PLOT_FRAC = 0.35

In [ ]:
import os
import sys
from pathlib import Path

work_dir = os.getcwd()
DEFAULT_BASE_DIR = os.path.normpath(os.path.join(work_dir, "..", ".."))

base_dir = Path(os.environ.get("BASE_DIR", DEFAULT_BASE_DIR)).resolve()

data_dir = base_dir / "data" / "mill"
model_dir = base_dir / "models"
plot_dir = base_dir / "plots"
results_dir = base_dir / "results"
for _d in (data_dir, model_dir, plot_dir, results_dir):
    os.makedirs(_d, exist_ok=True)

from src.utils import mill_io as mio

NameError: name 'data_dir' is not defined

In [ ]:
import numpy as np
import pandas as pd
from src.utils import run_registry as rr

rr.assert_current()
ledger = rr.run_ledger(work_dir)
data_store = rr.mill_data_store(data_dir)
model_store = rr.mill_model_store(model_dir)
plot_store = rr.mill_plot_store(plot_dir, ledger=ledger)
res_store = rr.mill_results_store(results_dir, ledger=ledger)

FEATURES, TARGET = mio.FEATURES, mio.TARGET

run_id = RUN_ID or data_store.latest_id("data_scored_stream")
if run_id is None:
    raise FileNotFoundError("Kein data_scored_stream -- bitte mill_model.ipynb ausfuehren.")
scored = data_store.load("data_scored_stream", run_id=run_id)
doe_scored = {w: data_store.load(f"data_scored_doe{w}", run_id=run_id) for w in ("1", "2")}
model_cfg = rr.RunConfig.from_document(scored.attrs["config"])
data_id = model_cfg.cfg["data_id"]
calibrations = pd.DatetimeIndex(
    [pd.Timestamp(t) for t in data_store.load("data_calibrations", run_id=data_id)])

ledger.bind("context", parents={"model": model_cfg.id})
print(f"model_id={model_cfg.id}  data_id={data_id}")
print(f"Produktion {len(scored)} Punkte, {scored.index[0]:%d.%m.%Y} bis "
      f"{scored.index[-1]:%d.%m.%Y} | {len(calibrations)} Kalibrierungen")

### Aufbereitung der Kontextsignale

Drei Arten von Größen stehen nebeneinander:

* **gemessen:** Walzenhöhen, Walzentemperatur, Materialfeuchte, Wassereindüsung;
* **aus der Wartung:** Stunden und Tonnen seit der letzten Kalibrierung — der
  Verschleiß wächst mit der durchgesetzten Menge, nicht mit der Kalenderzeit;
* **aus dem Betrieb:** Dauer des stationären Fensters, Stillstände je Woche und die
  Streuung der Betriebspunkte `rolling_var`. Letztere ist die Summe der gleitenden
  Varianzen aller Merkmale, jedes auf seine festen Anlagengrenzen skaliert — so
  zählt ein Prozent Sichterdrehzahl so viel wie ein Prozent Aufgabemenge.

In [ ]:
def clean_context(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col, (lo, hi) in PLAUSIBLE.items():
        if col in out:
            out[col] = out[col].where(out[col].between(lo, hi))
    for col, lo in CLIP_LOWER.items():
        if col in out:
            out[col] = out[col].clip(lower=lo)
    return out


def scale_features(df: pd.DataFrame) -> pd.DataFrame:
    lim = mio.TAGS.set_index("name")
    return pd.DataFrame({c: (df[c] - lim.at[c, "minimum"])
                         / (lim.at[c, "maximum"] - lim.at[c, "minimum"])
                         for c in FEATURES}, index=df.index)


def rolling_var(df: pd.DataFrame, days: float) -> pd.Series:
    return scale_features(df).rolling(f"{days}D", min_periods=3).var().sum(axis=1,
                                                                            min_count=1)


def stops_per_window(index: pd.DatetimeIndex, gap_h: float, days: float) -> pd.Series:
    gap = pd.Series(index, index=index).diff().dt.total_seconds().div(3600.0)
    return (gap > gap_h).astype(float).rolling(f"{days}D").sum()


def since_last(index: pd.DatetimeIndex, events: pd.DatetimeIndex,
               values: pd.Series = None) -> pd.Series:
    pos = events.searchsorted(index, side="right") - 1
    valid = pos >= 0
    out = pd.Series(np.nan, index=index)
    if values is None:
        dt = (index[valid] - events[pos[valid]]).total_seconds() / 3600.0
        out[valid] = dt
    else:
        at_event = np.interp(events.as_unit("ns").asi8, values.index.as_unit("ns").asi8,
                             values.to_numpy(float))
        out[valid] = values.to_numpy(float)[valid] - at_event[pos[valid]]
    return out


stream = clean_context(scored)
stream["abs_error"] = stream["model_error"].abs()
stream["roller_height"] = stream[[f"roller_height_{i}" for i in range(1, 5)]].mean(axis=1)
stream["rolling_var"] = rolling_var(stream, VAR_WIN_DAYS)
stream["stops_week"] = stops_per_window(stream.index, STOP_GAP_H, STOP_WIN_DAYS)
stream["hours_since_calib"] = since_last(stream.index, calibrations)
stream["tonnes_since_calib"] = since_last(stream.index, calibrations,
                                          stream["tonnage"].cummax())

_n_bad = {c: int((scored[c].notna() & stream[c].isna()).sum())
          for c in PLAUSIBLE if c in scored}
print("verworfen als unplausibel:", _n_bad)
print(stream[["feed_moisture", "water_injection", "rolling_var", "stops_week",
              "hours_since_calib", "tonnes_since_calib"]].describe().T
      .round(3).to_string())

## 1 Modellfehler und Streuung der Betriebspunkte

Ausgangspunkt der Masterarbeit: Hängt der Fehler daran, dass die Anlage gerade
unruhig gefahren wird? Die Farbe zeigt `rolling_var`. Ist der Fehler dort groß, wo
die Betriebspunkte stark streuen, wäre er ein Problem der Abdeckung, nicht der Zeit.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.utils import thesis_style as ts

w = ts.fig_width()
rng = np.random.default_rng(SEED)
DATE_FMT = mdates.DateFormatter("%m/%y")


def date_axis(ax, interval=6):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    ax.xaxis.set_major_formatter(DATE_FMT)


def mark_calibrations(ax, label=True):
    for i, t in enumerate(calibrations):
        ax.axvline(t, zorder=1, label="Kalibrierung" if (label and i == 0) else None,
                   **ts.vline("drift_event", lw=0.8))


_m = rng.random(len(stream)) < PLOT_FRAC
fig, ax = plt.subplots(figsize=(w, w * 0.36))
sc = ax.scatter(stream.index[_m], stream["model_error_norm"][_m],
                c=stream["rolling_var"][_m], cmap="viridis", s=6, alpha=0.7,
                marker=ts.MODEL_MARKER, rasterized=True)
cb = fig.colorbar(sc, ax=ax, pad=0.01)
cb.set_label(f"BP-Streuung ({VAR_WIN_DAYS} d)")
ax.set_ylabel("Norm. Modellfehler")
ax.set_xlabel("Zeit")
ax.set_xlim(stream.index[0], stream.index[-1])
ax.grid(True, alpha=0.2)
date_axis(ax)
plot_store.save_figure(fig, "context_error_variance", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

_ok = stream[["rolling_var", "abs_error"]].dropna()
R_VAR_ERR = float(_ok.corr(method="spearman").iloc[0, 1])
print(f"Spearman(rolling_var, |e|) = {R_VAR_ERR:+.3f} auf {len(_ok)} Punkten")

## 2 Stellgrößen über der Zeit

Gleitendes Mittel über `TREND_WIN_DAYS` Tage mit 95-%-Konfidenzband des Mittelwerts,
darunter der Modellfehler im selben Fenster. Das Exposé las daraus einen Wechsel im
Vorzeichen des Zusammenhangs zwischen Aufgabemenge und Fehler ab. Hier steht er als
gleitende Korrelation über `CORR_WIN_DAYS` Tage, statt aus zwei Bestgeraden.

In [ ]:
TREND_FEATURES = ["feed_rate", "classifier_speed", "tension_pressure",
                  "mill_fan_speed", "main_drive_speed"]
LABEL = {"feed_rate": "Aufgabe [t/h]", "classifier_speed": "Sichter [%]",
         "tension_pressure": "Spanndruck [bar]", "mill_fan_speed": "Ventilator [%]",
         "main_drive_speed": "Hauptantrieb [%]", "model_error": "Fehler [kWh/t]"}


def rolling_band(s: pd.Series, days: float):
    r = s.rolling(f"{days}D", min_periods=5)
    mean, half = r.mean(), 1.96 * r.std() / np.sqrt(r.count())
    return mean, mean - half, mean + half


_cols = TREND_FEATURES + ["model_error"]
fig, axes = plt.subplots(len(_cols), 1, sharex=True, figsize=(w, w * 1.05),
                         gridspec_kw={"hspace": 0.12})
for ax, col in zip(axes, _cols):
    mean, lo, hi = rolling_band(stream[col], TREND_WIN_DAYS)
    role = "drift_afflicted" if col == "model_error" else "drift_free"
    ax.fill_between(mean.index, lo, hi, color=ts.C[role], alpha=0.25, lw=0)
    ax.plot(mean.index, mean, color=ts.C[role], lw=1.0)
    ax.set_ylabel(LABEL[col], fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    mark_calibrations(ax, label=False)
axes[-1].axhline(0, color=ts.C["box"], lw=0.6)
axes[-1].set_xlabel("Zeit")
axes[-1].set_xlim(stream.index[0], stream.index[-1])
date_axis(axes[-1])
fig.align_ylabels(axes)
plot_store.save_figure(fig, "context_setpoints", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

In [ ]:
_daily = stream[TREND_FEATURES + ["model_error"]].resample("D").mean()
rolling_corr = pd.DataFrame({
    c: _daily[c].rolling(CORR_WIN_DAYS, min_periods=CORR_WIN_DAYS // 2)
       .corr(_daily["model_error"]) for c in TREND_FEATURES})
_half_year = rolling_corr.resample("6MS").mean()
_half_year.index = _half_year.index.strftime("%m/%y")
print(f"Gleitende Korrelation ({CORR_WIN_DAYS} d) Stellgroesse / Fehler, Mittel je Halbjahr:")
print(_half_year.round(2).to_string())
SIGN_CHANGES = {c: int((np.sign(rolling_corr[c].dropna()).diff().abs() > 0).sum())
                for c in TREND_FEATURES}
print("Vorzeichenwechsel:", SIGN_CHANGES)

## 3 Kontextanalyse

Pearson misst den linearen Zusammenhang, Mutual Information (MI) auch den
nichtlinearen (MA Gl. 4.1/4.2). Beide Matrizen laufen über alle Signale, nicht nur
gegen den Fehler — so lassen sich die erwarteten Zusammenhänge (Spanndruck ↔
Walzenhöhe, Aufgabe ↔ Walzenhöhe) als Plausibilitätsprüfung mitlesen.

MI hat keine feste Obergrenze. Für die Abbildung wird sie deshalb über den
Informationskorrelationskoeffizienten $r_I = \sqrt{1 - e^{-2I}}$ (Linfoot) auf
$[0, 1]$ gebracht; bei normalverteilten Größen fällt er mit $|\rho|$ zusammen, und
ein Abstand zwischen beiden Matrizen zeigt einen nichtlinearen Zusammenhang.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

CONTEXT_SIGNALS = {
    "model_error": "Fehler $e$", "abs_error": "$|e|$",
    "feed_rate": "Aufgabe", "classifier_speed": "Sichter",
    "tension_pressure": "Spanndruck", "mill_fan_speed": "Ventilator",
    "main_drive_speed": "Hauptantrieb", "recirculation_damper_position": "Rezirk.-Klappe",
    "stack_damper_position": "Kaminklappe", "hotgas_generator_load": "Heissgas",
    "roller_height": "Walzenhöhe", "roller_temperature": "Walzentemp.",
    "feed_moisture": "Feuchte", "water_injection": "Wassereind.",
    "hours_since_calib": "h seit Kalib.", "tonnes_since_calib": "t seit Kalib.",
    "duration_min": "Fensterdauer", "stops_week": "Stillst./Woche",
    "rolling_var": "BP-Streuung",
}
ctx = stream[list(CONTEXT_SIGNALS)].dropna()
print(f"Kontextanalyse auf {len(ctx)} von {len(stream)} Punkten "
      "(Punkte vor der ersten Kalibrierung und mit Sensorfehlern fallen heraus)")

pearson = ctx.corr()
_X = ctx.to_numpy(float)
mi = pd.DataFrame(
    np.vstack([mutual_info_regression(_X, _X[:, j], random_state=SEED)
               for j in range(_X.shape[1])]),
    index=ctx.columns, columns=ctx.columns)
mi = (mi + mi.T) / 2

ranking = pd.DataFrame({"pearson_e": pearson["model_error"],
                        "pearson_abs_e": pearson["abs_error"],
                        "mi_e": mi["model_error"], "mi_abs_e": mi["abs_error"]}
                       ).drop(["model_error", "abs_error"]).sort_values(
                           "mi_e", ascending=False)
print(ranking.round(3).to_string())

**Lesehilfe zur MI-Matrix.** Alle Signale wechseln mit dem Betriebsregime, die
kNN-Schätzung der MI nimmt diese gemeinsame Zeitstruktur mit. Die absolute Höhe
ist deshalb durchweg hoch; aussagekräftig ist die Rangfolge innerhalb einer Zeile,
vor allem der Zeile des Fehlers.

In [ ]:
def heatmap(matrix: pd.DataFrame, title: str, name: str, *, cmap, vmin, vmax,
            fmt="{:+.2f}"):
    labels = [CONTEXT_SIGNALS[c] for c in matrix.columns]
    fig, ax = plt.subplots(figsize=(w, w * 0.9))
    im = ax.imshow(matrix.to_numpy(float), cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=6)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=6)
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = matrix.iat[i, j]
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=4,
                    color="white" if abs(v) > 0.6 * max(abs(vmin), abs(vmax)) else "black")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label(title, fontsize=7)
    plot_store.save_figure(fig, name, model_cfg, final=IS_FINAL,
                           savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                           archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
    plt.show()


heatmap(pearson, "Pearson $\\rho$", "context_pearson", cmap="RdBu_r", vmin=-1, vmax=1)
_ri = np.sqrt(1 - np.exp(-2 * mi.clip(lower=0).to_numpy()))
np.fill_diagonal(_ri, 1.0)
mi_corr = pd.DataFrame(_ri, index=mi.index, columns=mi.columns)
heatmap(mi_corr, "MI als $r_I$", "context_mi", cmap="viridis", vmin=0, vmax=1,
        fmt="{:.2f}")

## 4 Kalibrierungen und Modellgüte

Die Masterarbeit hielt Kalibrierungen, Wartungen und den Materialwechsel per
Augenschein gegen den Fehlerverlauf. Hier steht dasselbe Bild, dazu eine Zahl: Für
jede Kalibrierung wird der Fehler in den `CAL_WIN_DAYS` Tagen davor mit denen danach
verglichen — als Verschiebung des Mittelwerts (Bias) und des RMSE. Ob diese
Verschiebungen größer sind als an beliebigen Tagen, entscheidet ein Vergleich mit
`N_PERM` zufällig gezogenen Zeitpunkten desselben Stroms.

Dieselbe Rechnung läuft für `rolling_var`: Die Masterarbeit beobachtete, dass nach
einer Kalibrierung unruhiger gefahren wird.

In [ ]:
rmse_roll = np.sqrt((stream["model_error"] ** 2)
                    .rolling(f"{RMSE_WIN_DAYS}D", min_periods=5).mean())
daily_height = stream[[f"roller_height_{i}" for i in range(1, 5)]].resample("D").mean()

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(w, w * 0.75),
                         gridspec_kw={"hspace": 0.1})
for i, col in enumerate(daily_height.columns):
    axes[0].plot(daily_height.index, daily_height[col], lw=0.8,
                 color=ts.CATEGORICAL[i], label=f"Walze {i + 1}")
axes[0].set_ylabel("Walzenhöhe [mm]", fontsize=8)
axes[0].legend(frameon=False, fontsize=7, ncol=5, loc="lower center",
               bbox_to_anchor=(0.5, 1.0))
axes[1].plot(rmse_roll.index, rmse_roll, color=ts.C["drift_afflicted"], lw=1.0)
axes[1].set_ylabel(f"RMSE {RMSE_WIN_DAYS} d [kWh/t]", fontsize=8)
axes[2].plot(stream.index, stream["rolling_var"], color=ts.C["box"], lw=0.8)
axes[2].set_ylabel(f"BP-Streuung {VAR_WIN_DAYS} d", fontsize=8)
axes[2].set_xlabel("Zeit")
for ax in axes:
    mark_calibrations(ax, label=(ax is axes[1]))
    ax.grid(True, alpha=0.2)
    ax.tick_params(labelsize=7)
axes[1].legend(frameon=False, fontsize=7, loc="upper right")
axes[-1].set_xlim(stream.index[0], stream.index[-1])
date_axis(axes[-1])
fig.align_ylabels(axes)
plot_store.save_figure(fig, "context_calibrations", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

In [ ]:
def window_shift(times: pd.DatetimeIndex, values: np.ndarray, t0, days: float,
                 min_points: int):
    """Mittelwert, RMSE und Varianz nach minus vor ``t0`` im Fenster ``days``."""
    half = pd.Timedelta(days=days)
    a, b, c = times.searchsorted([t0 - half, t0, t0 + half])
    before, after = values[a:b], values[b:c]
    before, after = before[np.isfinite(before)], after[np.isfinite(after)]
    if len(before) < min_points or len(after) < min_points:
        return None
    return {"d_mean": after.mean() - before.mean(),
            "d_rmse": np.sqrt(np.mean(after ** 2)) - np.sqrt(np.mean(before ** 2))}


def shifts_at(times, values, moments):
    rows = {t: window_shift(times, values, t, CAL_WIN_DAYS, CAL_MIN_POINTS)
            for t in moments}
    return pd.DataFrame({t: r for t, r in rows.items() if r is not None}).T


_t, _e = stream.index, stream["model_error"].to_numpy(float)
_v = stream["rolling_var"].to_numpy(float)
cal_in = calibrations[(calibrations > _t[0]) & (calibrations < _t[-1])]
cal_err = shifts_at(_t, _e, cal_in)
cal_var = shifts_at(_t, _v, cal_in)

_lo, _hi = _t[0] + pd.Timedelta(days=CAL_WIN_DAYS), _t[-1] - pd.Timedelta(days=CAL_WIN_DAYS)
_rand = pd.DatetimeIndex(rng.uniform(_lo.value, _hi.value, N_PERM).astype("int64")).floor("s")
rand_err = shifts_at(_t, _e, _rand)
rand_var = shifts_at(_t, _v, _rand)


def exceedance(observed: pd.Series, reference: pd.Series, n_draws: int) -> float:
    """Anteil zufaelliger Stichproben gleicher Groesse mit mindestens so grossem
    mittlerem Betrag -- das Vergleichsniveau fuer die Kalibrierungen."""
    ref = np.abs(reference.to_numpy(float))
    obs = float(np.mean(np.abs(observed)))
    draws = rng.choice(ref, size=(n_draws, len(observed)), replace=True).mean(axis=1)
    return float(((draws >= obs).sum() + 1) / (n_draws + 1))


CAL_TEST = pd.DataFrame({
    "Kalibrierung": [cal_err["d_mean"].abs().mean(), cal_err["d_rmse"].abs().mean(),
                     cal_var["d_mean"].abs().mean()],
    "Zufall": [rand_err["d_mean"].abs().mean(), rand_err["d_rmse"].abs().mean(),
               rand_var["d_mean"].abs().mean()],
    "p": [exceedance(cal_err["d_mean"], rand_err["d_mean"], N_PERM),
          exceedance(cal_err["d_rmse"], rand_err["d_rmse"], N_PERM),
          exceedance(cal_var["d_mean"], rand_var["d_mean"], N_PERM)],
}, index=["|d Bias| [kWh/t]", "|d RMSE| [kWh/t]", "|d rolling_var|"])
print(f"{len(cal_err)} von {len(calibrations)} Kalibrierungen auswertbar "
      f"(+/-{CAL_WIN_DAYS} d, je Seite >= {CAL_MIN_POINTS} Punkte)")
print(CAL_TEST.to_string(float_format=lambda v: f"{v:.4f}"))
print("\nje Kalibrierung:")
print(cal_err.rename(columns={"d_mean": "d_bias"})
      .join(cal_var[["d_mean"]].rename(columns={"d_mean": "d_rolling_var"}))
      .round(3).to_string())

## 5 Driftart

Zwei Vergleiche mit dem Zwei-Stichproben-Kolmogorow-Smirnow-Test, je Merkmal und für
die Zielgröße:

* zwischen den Datensätzen DOE1, DOE2 und Produktion (MA 4.3);
* innerhalb der Produktion, erste gegen zweite Hälfte des Zeitraums.

Ändert sich die Verteilung der Merkmale, liegt Kovariatenverschiebung vor; ändert
sich die der Zielgröße, eine Verschiebung der A-priori-Verteilung. Ob sich
$P(y\mid x)$ ändert — echter Concept Drift —, zeigt der Test nicht; dafür steht
Abschnitt 6.

In [ ]:
from scipy.stats import ks_2samp

_mid = stream.index[0] + (stream.index[-1] - stream.index[0]) / 2
KS_SETS = {"DOE1": doe_scored["1"], "DOE2": doe_scored["2"], "Produktion": stream,
           "P1": stream.loc[:_mid], "P2": stream.loc[_mid:]}
KS_PAIRS = [("DOE1", "DOE2"), ("DOE1", "Produktion"), ("DOE2", "Produktion"),
            ("P1", "P2")]
ks = pd.DataFrame({f"{a}-{b}": {c: ks_2samp(KS_SETS[a][c].dropna(),
                                             KS_SETS[b][c].dropna()).pvalue
                                 for c in FEATURES + [TARGET]}
                   for a, b in KS_PAIRS})
print("p-Werte:")
print(ks.to_string(float_format=lambda v: f"{v:.1e}"))
KS_REJECTED = (ks < KS_ALPHA).sum()
print(f"\nabgelehnt bei alpha={KS_ALPHA}:")
print(KS_REJECTED.to_string())

## 6 Cluster: lokaler oder globaler Drift?

Die 18 Betriebspunkte aus DOE1 sind die Zentren (MA 4.3.1: ein Cluster je DOE-Punkt). Jeder Produktionspunkt gehört zum
nächstgelegenen Zentrum (euklidisch, Merkmale auf ihre festen Anlagengrenzen
skaliert). Weil die Produktion den DOE-Raum nur teilweise trifft, wird zusätzlich
unterschieden, ob ein Punkt **innerhalb** des Clusters liegt: näher als
`RADIUS_FRAC` mal der Abstand des Zentrums zu seinem nächsten Nachbarzentrum.

Nur diese Punkte tragen die Aussage der Masterarbeit (Tab. 4.2/4.3): Weicht die
Zielgröße bei nahezu gleichem Betriebspunkt vom DOE1-Wert ab, hat sich $P(y\mid x)$
geändert.

In [ ]:
_doe1 = doe_scored["1"]
centers = scale_features(_doe1).to_numpy(float)
center_tse = _doe1[TARGET].to_numpy(float)
_cc = np.linalg.norm(centers[:, None] - centers[None], axis=2)
np.fill_diagonal(_cc, np.inf)
radius = RADIUS_FRAC * _cc.min(axis=1)


def assign(df: pd.DataFrame):
    d = np.linalg.norm(scale_features(df).to_numpy(float)[:, None] - centers[None], axis=2)
    k = d.argmin(axis=1)
    dist = d[np.arange(len(k)), k]
    return k, dist, dist <= radius[k]


stream["cluster"], stream["cluster_dist"], stream["inside"] = assign(stream)
_k2, _d2, _in2 = assign(doe_scored["2"])

rows = []
for k in range(len(centers)):
    sel = stream["cluster"] == k
    ins = sel & stream["inside"]
    rows.append({
        "tse_doe1": center_tse[k], "radius": radius[k],
        "n_nearest": int(sel.sum()), "n_inside": int(ins.sum()),
        "n_inside_doe2": int(((_k2 == k) & _in2).sum()),
        "dist_nearest": stream.loc[sel, "cluster_dist"].mean(),
        "tse_dev_nearest": stream.loc[sel, TARGET].mean() - center_tse[k],
        "tse_dev_inside": (stream.loc[ins, TARGET].mean() - center_tse[k]
                           if ins.any() else np.nan),
        "error_inside": stream.loc[ins, "model_error"].mean() if ins.any() else np.nan,
    })
clusters = pd.DataFrame(rows)
print(clusters.round(3).to_string())

N_INSIDE = int(stream["inside"].sum())
N_CLUSTERS_INSIDE = int((clusters["n_inside"] > 0).sum())
print(f"\n{N_INSIDE} Produktionspunkte ({100 * N_INSIDE / len(stream):.1f} %) liegen in "
      f"{N_CLUSTERS_INSIDE} der {len(centers)} DOE1-Cluster")
print(f"mittlerer Abstand zum naechsten Zentrum {stream['cluster_dist'].mean():.3f}, "
      f"mittlerer Clusterradius {radius.mean():.3f}")

_err = stream["model_error"]
_grp = _err.groupby(stream["cluster"])
ETA2 = float(((_grp.mean() - _err.mean()) ** 2 * _grp.size()).sum()
             / ((_err - _err.mean()) ** 2).sum())
print(f"Anteil der Fehlervarianz, der auf die Clusterzugehoerigkeit entfaellt: "
      f"eta^2 = {ETA2:.3f}")

Drei Bilder dazu. Die Cluster sind nach der Zielgröße ihres DOE1-Zentrums sortiert
und entlang eines Farbverlaufs eingefärbt, wie im Exposé (Abb. 6):

1. der Modellfehler, eingefärbt nach Cluster, mit den Kalibrierungen;
2. die Belegung der Cluster über der Zeit (MA Abb. 4.9);
3. der Fehler je Cluster über der Zeit (MA Abb. 4.10). Laufen die Zeilen gemeinsam,
   ist der Drift global; weicht eine Zeile allein ab, lokal.

In [ ]:
order = np.argsort(center_tse)
rank = np.empty_like(order)
rank[order] = np.arange(len(order))
cmap = plt.get_cmap("viridis", len(centers))
stream["cluster_rank"] = rank[stream["cluster"].to_numpy()]

fig, ax = plt.subplots(figsize=(w, w * 0.36))
sc = ax.scatter(stream.index[_m], stream["model_error_norm"][_m],
                c=stream["cluster_rank"][_m], cmap=cmap, vmin=-0.5,
                vmax=len(centers) - 0.5, s=6, alpha=0.75, marker=ts.MODEL_MARKER,
                rasterized=True)
mark_calibrations(ax)
cb = fig.colorbar(sc, ax=ax, pad=0.01)
cb.set_label("Cluster, nach spez. Energie")
ax.set_ylabel("Norm. Modellfehler")
ax.set_xlabel("Zeit")
ax.set_xlim(stream.index[0], stream.index[-1])
ax.legend(frameon=False, fontsize=7, loc="lower left")
ax.grid(True, alpha=0.2)
date_axis(ax)
plot_store.save_figure(fig, "context_cluster_error", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

In [ ]:
_occ = (pd.get_dummies(stream["cluster_rank"]).astype(float)
        .reindex(columns=range(len(centers)), fill_value=0.0)
        .rolling(f"{OCC_WIN_DAYS}D").sum())
_occ = _occ.div(_occ.sum(axis=1), axis=0).resample("D").last().dropna()

fig, ax = plt.subplots(figsize=(w, w * 0.36))
ax.stackplot(_occ.index, _occ.T.to_numpy(), colors=[cmap(i) for i in range(len(centers))],
             linewidth=0)
ax.set_ylim(0, 1)
ax.set_ylabel(f"Anteil ({OCC_WIN_DAYS} d)")
ax.set_xlabel("Zeit")
ax.set_xlim(_occ.index[0], _occ.index[-1])
date_axis(ax)
plot_store.save_figure(fig, "context_cluster_occupancy", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

_top = _occ.max(axis=1)
print(f"Anteil des staerksten Clusters im {OCC_WIN_DAYS}-Tage-Fenster: Median "
      f"{_top.median():.2f}, 90-%-Quantil {_top.quantile(0.9):.2f}")
OCC_TOP3 = float(np.sort(_occ.to_numpy(), axis=1)[:, -3:].sum(axis=1).mean())
print(f"Die drei staerksten Cluster tragen im Mittel {100 * OCC_TOP3:.0f} % der Punkte")

In [ ]:
_weekly = (stream.groupby([pd.Grouper(freq="W"), "cluster_rank"])["model_error"]
           .mean().unstack())
_scale = 0.45 / np.nanpercentile(np.abs(_weekly.to_numpy()), 95)

fig, ax = plt.subplots(figsize=(w, w * 0.8))
for r in range(len(centers)):
    if r not in _weekly:
        continue
    y = _weekly[r]
    ax.axhline(r, color=ts.C["box"], lw=0.4, alpha=0.5)
    ax.fill_between(y.index, r, r + _scale * y, color=cmap(r), alpha=0.8, lw=0)
    ax.plot(y.index, r + _scale * y, ".", ms=1.5, color=cmap(r))
mark_calibrations(ax, label=False)
ax.set_yticks(range(len(centers)))
ax.set_yticklabels([f"C{order[r] + 1} ({center_tse[order[r]]:.0f})"
                    for r in range(len(centers))], fontsize=6)
ax.set_ylabel("Cluster (spez. Energie des Zentrums [kWh/t])", fontsize=8)
ax.set_xlabel("Zeit")
ax.set_ylim(-1, len(centers))
ax.set_xlim(stream.index[0], stream.index[-1])
date_axis(ax)
plot_store.save_figure(fig, "context_cluster_ridge", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

_wk = _weekly.dropna(axis=1, thresh=20)
_cr = _wk.corr(min_periods=10).to_numpy()
CLUSTER_CORR = float(np.nanmean(_cr[np.triu_indices_from(_cr, 1)]))
print(f"mittlere Korrelation der Wochenfehler zwischen {_wk.shape[1]} Clustern: "
      f"{CLUSTER_CORR:+.2f} (nahe 1: globaler Drift)")

## 7 Gradientenbasierte Einordnung

Über die geglättete Ableitung des gleitenden RMSE lässt sich unterscheiden, ob der
Fehler springt (sudden) oder wandert (incremental). Die Schwellen sind Vielfache der
robusten Streuung (MAD) der Ableitung. Das bleibt eine begründete Einordnung, kein
Nachweis.

In [ ]:
from scipy.signal import savgol_filter

_rd = rmse_roll.resample("D").mean().interpolate(limit_direction="both")
grad = pd.Series(savgol_filter(_rd.to_numpy(float), int(SAVGOL_WIN_DAYS) | 1,
                               SAVGOL_ORDER, deriv=1), index=_rd.index)
_g = grad.to_numpy()
_mad = float(np.median(np.abs(_g - np.median(_g)))) * 1.4826 or float(np.std(_g))
sudden = grad.index[np.abs(_g) > 4 * _mad]
incremental = grad.index[(np.abs(_g) > 1.2 * _mad) & (np.abs(_g) <= 4 * _mad)]
print(f"Tage mit sprunghafter Fehleraenderung: {len(sudden)} "
      f"({100 * len(sudden) / len(grad):.1f} %)")
print(f"Tage mit inkrementeller Aenderung:     {len(incremental)} "
      f"({100 * len(incremental) / len(grad):.1f} %)")

_near = [t for t in sudden if len(cal_in) and np.min(np.abs((cal_in - t).days)) <= CAL_WIN_DAYS]
print(f"davon sprunghaft innerhalb +/-{CAL_WIN_DAYS} d um eine Kalibrierung: "
      f"{len(_near)} von {len(sudden)}")

fig, ax = plt.subplots(figsize=(w, w * 0.3))
ax.plot(grad.index, grad.values, color=ts.C["drift_afflicted"], lw=0.9)
for s in (4 * _mad, -4 * _mad):
    ax.axhline(s, **ts.line("box", linewidth=0.8, linestyle="--"))
mark_calibrations(ax)
ax.set_ylabel(r"$\mathrm{d}\,\mathrm{RMSE}/\mathrm{d}t$")
ax.set_xlabel("Zeit")
ax.set_xlim(grad.index[0], grad.index[-1])
ax.legend(frameon=False, fontsize=7, loc="upper right")
ax.grid(True, alpha=0.2)
date_axis(ax)
plot_store.save_figure(fig, "context_gradient", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

## 8 Ereignisreferenz für die Detektion

Die Kalibrierungen im Zeitraum des Stroms, abgebildet auf den ersten Betriebspunkt
am oder nach dem Zeitpunkt. *mill_detection* liest sie über `CONTEXT_ID`.

In [ ]:
ctx_cfg = model_cfg.compose(context={
    "plausible": PLAUSIBLE, "clip_lower": CLIP_LOWER, "var_win_days": VAR_WIN_DAYS,
    "stop_gap_h": STOP_GAP_H, "stop_win_days": STOP_WIN_DAYS,
    "rmse_win_days": RMSE_WIN_DAYS, "cal_win_days": CAL_WIN_DAYS,
    "cal_min_points": CAL_MIN_POINTS, "n_perm": N_PERM, "radius_frac": RADIUS_FRAC,
    "savgol": [SAVGOL_WIN_DAYS, SAVGOL_ORDER], "seed": SEED})

event_idx = sorted({int(i) for i in stream.index.searchsorted(cal_in)
                    if i < len(stream)})
events = {"timestamps": [str(t) for t in cal_in], "indices": event_idx,
          "source": "data_calibrations", "data_id": data_id}
model_store.save(events, "context_events", ctx_cfg)
print(f"ctx_id={ctx_cfg.id}: {len(event_idx)} Ereignisse")

## Ergebnisse speichern

In [ ]:
from src.utils import results_export as rx

(rx.ResultDoc()
 .integer("n_points", len(stream))
 .integer("n_context_points", len(ctx))
 .num("spearman_rolling_var_abs_error", R_VAR_ERR, 3)
 .set("rolling_corr_sign_changes", SIGN_CHANGES)
 .set("context_ranking", {k: {c: float(v[c]) for c in ranking.columns}
                          for k, v in ranking.iterrows()})
 .integer("n_calibrations", len(calibrations))
 .integer("n_calibrations_evaluated", len(cal_err))
 .set("calibration_test", {k: {c: float(v[c]) for c in CAL_TEST.columns}
                           for k, v in CAL_TEST.iterrows()})
 .set("ks_pvalues", {k: {c: float(v) for c, v in ks[k].items()} for k in ks.columns})
 .set("ks_rejected", {k: int(v) for k, v in KS_REJECTED.items()})
 .integer("n_inside", N_INSIDE)
 .integer("n_clusters_inside", N_CLUSTERS_INSIDE)
 .num("eta2_cluster", ETA2, 3)
 .num("cluster_weekly_corr", CLUSTER_CORR, 3)
 .num("occupancy_top3", OCC_TOP3, 3)
 .set("clusters", clusters.round(4).to_dict(orient="index"))
 .integer("n_sudden_days", len(sudden))
 .integer("n_incremental_days", len(incremental))
 .integer("n_sudden_near_calibration", len(_near))
 .set("event_indices", event_idx)
 .save(res_store, "context", ctx_cfg, final=IS_FINAL,
       parents={"model": model_cfg.id}))

In [ ]:
from src.utils import latex_export as lx

mx = lx.MacroExport("automatisch erzeugt aus mill_context.ipynb - nicht manuell editieren")
mx.comment("Streuung der Betriebspunkte")
mx.num("millCtxRhoVarErr", R_VAR_ERR, 2)
mx.comment("Kontextanalyse: Pearson mit dem Modellfehler")
for _k, _tok in (("feed_moisture", "Moisture"), ("roller_height", "RollerHeight"),
                 ("tonnes_since_calib", "TonnesSinceCalib"),
                 ("rolling_var", "RollingVar"), ("stops_week", "Stops")):
    mx.num(f"millCtxPearson{_tok}", float(ranking.at[_k, "pearson_e"]), 2)
mx.comment("Kalibrierungen")
mx.integer("millCtxNCalib", len(calibrations))
mx.integer("millCtxNCalibEval", len(cal_err))
mx.integer("millCtxCalWinDays", CAL_WIN_DAYS)
mx.num("millCtxCalBias", float(CAL_TEST.iat[0, 0]), 2)
mx.num("millCtxRandBias", float(CAL_TEST.iat[0, 1]), 2)
mx.num("millCtxCalBiasP", float(CAL_TEST.iat[0, 2]), 4)
mx.num("millCtxCalRmseP", float(CAL_TEST.iat[1, 2]), 4)
mx.num("millCtxCalVarP", float(CAL_TEST.iat[2, 2]), 3)
mx.comment("Driftart (KS-Test)")
mx.integer("millCtxKsN", len(FEATURES) + 1)
for _pair, _tok in zip(ks.columns, ("DoeOneDoeTwo", "DoeOneProd", "DoeTwoProd", "Halves")):
    mx.integer(f"millCtxKsRej{_tok}", int(KS_REJECTED[_pair]))
mx.comment("Cluster")
mx.integer("millCtxNClusters", len(centers))
mx.integer("millCtxNInside", N_INSIDE)
mx.integer("millCtxNClustersInside", N_CLUSTERS_INSIDE)
mx.num("millCtxEtaSq", ETA2, 2)
mx.num("millCtxClusterCorr", CLUSTER_CORR, 2)
mx.num("millCtxOccTopThreePct", 100 * OCC_TOP3, 0)
mx.comment("Gradientenbasierte Einordnung")
mx.integer("millCtxNSuddenDays", len(sudden))
mx.integer("millCtxNIncrementalDays", len(incremental))
mx.save(res_store, "context", ctx_cfg, final=IS_FINAL)

In [ ]:
print(f"ctx_cfg.id = {ctx_cfg.id}   -> CONTEXT_ID in mill_detection")